# EuroSAT Few-Shot Benchmark

Four image models (ResNet-18, ConvNeXt-Tiny, ViT-B/16, DINO ViT-S/16), each **untrained**, **ImageNet-pretrained** and **fine-tuned**, classify satellite land cover from 1, 5 or 10 labeled images per class, with and without augmenting those images. Plan: `research_info.md`.

**How to use:** edit Section 1, then run all cells. Start with `MODE = 'smoke'`.


## 1 · Settings
The only cell to edit.

In [ ]:
# Where to run
TARGET = 'local'        # 'local' or 'colab_gpu'
STORAGE_DIR = ''        # data, cache, weights and results ('' = this repository, or Google Drive on Colab)
CODE_DIR = ''           # folder that contains src/ ('' = find it automatically)
MANIFEST_FILE = ''      # splits_manifest.json ('' = <storage>/splits_manifest.json)

# What to run
MODE = 'smoke'          # 'smoke' = quick code check, 'full' = the experiment
MODELS = 'all'          # or a subset, e.g. 'resnet18,dino_vits16'
BASELINES = ['untrained', 'pretrained', 'finetuned']
TRACKS = [1, 2, 3]      # 1 models, 2 all augmentations together, 3 each augmentation alone
VIEW_SWEEP = True       # extra 1-shot run with 1, 2, 4 and 8 augmented views (figure F9)

# Augmentation strength on the 224x224 input (defaults: src/eurosat/config.py)
AUGMENTATION = {
    'flip': {}, 'rotation': {},  # no strength setting
    'scale_up': {'min_zoom': 1.1, 'max_zoom': 1.3},
    'brightness_contrast': {'min_change': 0.1, 'max_change': 0.3},
    'noise': {'std': 0.05},
    'blur': {'min_sigma': 2.5, 'max_sigma': 3.5},
    'erasing': {'min_size_fraction': 0.2, 'max_size_fraction': 0.3, 'fill': 0.5},
    'cutout': {'min_size_fraction': 0.2, 'max_size_fraction': 0.3, 'fill': 0.0},
}

# Display
SHOW_DETAILS = False    # also show every data-check table and the supplementary figures and tables
REPORT_RESULTS = ''     # path to a saved results.json: show its results without computing anything


## 2 · Setup
Connects storage, loads the project code and shows what this run will do.

In [ ]:
import os
import sys
from pathlib import Path

# Automated runs (scripts/execute_notebook.py) can override these with EUROSAT_* variables.
TARGET = os.environ.get('EUROSAT_TARGET', TARGET)
STORAGE_DIR = os.environ.get('EUROSAT_STORAGE', STORAGE_DIR)
CODE_DIR = os.environ.get('EUROSAT_CODE_DIR', CODE_DIR)

if TARGET == 'colab_gpu':  # Storage on Google Drive; images are extracted to Colab's fast local disk.
    from google.colab import drive
    drive.mount('/content/drive')
    parent = Path('/content/drive/MyDrive/ColabNotebooks/cs7643')
    STORAGE = Path(STORAGE_DIR or next((p for p in [parent / 'EuroSATLowData'] if p.exists()),
                                       parent / 'EuroSATRepresentationBenchmark'))
    if STORAGE_DIR and not STORAGE.is_dir():
        raise FileNotFoundError('STORAGE_DIR not found on Google Drive: check the My Drive shortcut')
    SCRATCH = Path('/content/eurosat-representation-benchmark')
else:  # Local: everything under the repository unless STORAGE_DIR says otherwise.
    here = Path.cwd()
    STORAGE = Path(STORAGE_DIR or next((p for p in [here, *here.parents] if (p / 'pyproject.toml').exists()), here))
    SCRATCH = STORAGE / 'data/extracted'
STORAGE.mkdir(parents=True, exist_ok=True)

places = [Path(CODE_DIR)] if CODE_DIR else [Path.cwd(), *Path.cwd().parents, Path('/content/eurosat-representation-benchmark'),
                                           STORAGE / 'code']
code = next((p for p in places if (p / 'src/eurosat/__init__.py').is_file()), None)
if code is None:
    raise FileNotFoundError('Project code not found: set CODE_DIR to the repository, or copy src/ into <storage>/code/')
sys.path.insert(0, str(code / 'src'))

from eurosat import notebook as nb
run = nb.Run.from_settings(globals())


## 3 · Data check
Verifies the dataset and selects the fixed episodes.

In [ ]:
run.check_data()


## 4 · Fine-tune
Trains the fine-tuned baseline. Models already trained are skipped.

In [ ]:
run.fine_tune()


## 5 · Run
Embeddings and classifiers for every baseline × model × condition, then the report figures and tables. Saved work is reused, so a stopped run resumes.

In [ ]:
run.run()


## 6 · Results: models and baselines
T1, F2, F3, F11 (figure_n_table_sug.md).

In [ ]:
run.show_models()


## 7 · Results: augmentation
F6, F8, T3.

In [ ]:
run.show_augmentation()


## 8 · Checks and files
Classifier checks, labels used (T4) and where the outputs are saved.

In [ ]:
run.show_checks()
